In [1]:
import torch

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "w11wo/indonesian-roberta-base-sentiment-classifier"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

label_map = model.config.id2label
print("Label mapping:", label_map)

def batch_predict(texts, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=1).tolist()
        results.extend([label_map[p] for p in preds])
    return results


sample_texts = [
    "aku sangat setuju dengan kebijakan ini",  
    "saya kecewa dengan pelayanan",           
    "biasa saja, tidak terlalu buruk",         
]
preds = batch_predict(sample_texts)
for text, label in zip(sample_texts, preds):
    print(f"{text} → {label}")



c:\Users\zam\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Label mapping: {0: 'positive', 1: 'neutral', 2: 'negative'}
aku sangat setuju dengan kebijakan ini → positive
saya kecewa dengan pelayanan → negative
biasa saja, tidak terlalu buruk → positive


In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Load model & tokenizer
MODEL = "w11wo/indonesian-roberta-base-sentiment-classifier"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

# Mapping label langsung dari config
label_map = model.config.id2label

def batch_predict(texts, batch_size=32):
    results = []
    # tqdm kasih progress bar
    for i in tqdm(range(0, len(texts), batch_size), desc="Labeling progress"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=1).tolist()
        results.extend([label_map[p] for p in preds])
    return results



c:\Users\zam\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:

df = pd.read_csv("tweets_texts.csv")

df["sentiment"] = batch_predict(df["text"].astype(str).tolist(), batch_size=32)

df.to_csv("tweets_with_sentiment.csv", index=False, encoding="utf-8-sig")
print("✅ Labeling selesai → hasil disimpan di tweets_with_sentiment.csv")

Labeling progress: 100%|██████████| 168/168 [09:48<00:00,  3.50s/it]

✅ Labeling selesai → hasil disimpan di tweets_with_sentiment.csv
